In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/drive/MyDrive/Universidad/Investigacion/dataframe_balanced.csv")
df

,id,problem_id,question,answer,tag
0,1,1,Write a JavaScript function that returns the f...,function factorial(n) {\n let result = 1;\n ...,0
1,2,1,Write a JavaScript function that returns the f...,function factorial(n) {\n let i = 1;\n let r...,0
2,3,1,Write a JavaScript function that returns the f...,function factorial(n) {\n let result = 1;\n ...,0
3,4,1,Write a JavaScript function that returns the f...,function factorial(n) {\n let i = 2;\n let r...,0
4,5,1,Write a JavaScript function that returns the f...,function factorial(n) {\n let result = 1;\n ...,0
...,...,...,...,...,...
49491,49492,29,Write a JavaScript function that given a strin...,function capitalizeLetters(str) {\n let resul...,7
49492,49493,29,Write a JavaScript function that given a strin...,function capitalizeLetters(str) {\n let i = s...,7
49493,49494,29,Write a JavaScript function that given a strin...,function capitalizeLetters(str) {\n let resul...,7
49494,49495,29,Write a JavaScript function that given a strin...,function capitalizeLetters(str) {\n let i = 0...,7


In [3]:
import re
import textwrap
from typing import Dict, Optional


def find_matching_brace(s: str, start_idx: int) -> int:
    brace = 1
    i = start_idx
    while i < len(s) and brace > 0:
        if s[i] == "{":
            brace += 1
        elif s[i] == "}":
            brace -= 1
        i += 1
    return i


def extract_sections_from_function(code: str) -> Dict[str, str]:
    func_pattern = re.compile(
        r"function\s+([A-Za-z0-9_$]+)\s*\(([^)]*)\)\s*\{", re.DOTALL
    )
    m = func_pattern.search(code)
    if not m:
        raise ValueError("No se encontró una función válida en el código.")

    name = m.group(1)
    params = m.group(2).strip()
    body_start = m.end()
    func_end = find_matching_brace(code, body_start)
    full_func = code[m.start() : func_end]

    func_body = code[body_start : func_end - 1]

    while_pattern = re.compile(r"while\s*\((.*?)\)\s*\{", re.DOTALL)
    wm = while_pattern.search(func_body)

    if not wm:
        return {
            "answer": textwrap.dedent(full_func).strip(),
            "function_name": name,
            "function_params": params,
            "initial": "",
            "transformation": "",
            "js": "",
            "final": "",
        }

    while_rel_start = wm.start()
    while_rel_end = wm.end()

    condition = wm.group(1).strip()

    initial_raw = func_body[:while_rel_start]
    initial = textwrap.dedent(initial_raw).strip() or ""

    transform_body_start_abs = body_start + while_rel_end
    transform_body_end_abs = find_matching_brace(code, transform_body_start_abs)
    transformation_raw = code[transform_body_start_abs : transform_body_end_abs - 1]
    transformation = textwrap.dedent(transformation_raw).strip() or ""

    final_raw = code[transform_body_end_abs : func_end - 1]
    final = textwrap.dedent(final_raw).strip() or ""

    return {
        "answer": textwrap.dedent(full_func).strip(),
        "function_name": name,
        "function_params": params,
        "initial": initial,
        "transformation": transformation,
        "js": condition,
        "final": final,
    }

In [4]:
extracted_data = df['answer'].apply(extract_sections_from_function)

extracted_df = pd.DataFrame(extracted_data.tolist())

temp_df = df.drop(columns=['answer'])

result_df = temp_df.join(extracted_df)

cols = result_df.columns.tolist()
cols.remove('tag')
cols.append('tag')
result_df = result_df[cols]

result_df

,id,problem_id,question,answer,function_name,function_params,initial,transformation,js,final,tag
0,1,1,Write a JavaScript function that returns the f...,function factorial(n) {\n let result = 1;\n ...,factorial,n,let result = 1;\nlet i = 1;,result *= i;\ni++;,i <= n,return result;,0
1,2,1,Write a JavaScript function that returns the f...,function factorial(n) {\n let i = 1;\n let r...,factorial,n,let i = 1;\nlet result = 1;,result *= i;\ni++;,i <= n,return result;,0
2,3,1,Write a JavaScript function that returns the f...,function factorial(n) {\n let result = 1;\n ...,factorial,n,let result = 1;\nlet i = 2;,result *= i;\ni++;,i <= n,return result;,0
3,4,1,Write a JavaScript function that returns the f...,function factorial(n) {\n let i = 2;\n let r...,factorial,n,let i = 2;\nlet result = 1;,result *= i;\ni++;,i <= n,return result;,0
4,5,1,Write a JavaScript function that returns the f...,function factorial(n) {\n let result = 1;\n ...,factorial,n,let result = 1;\nlet i = 1;,result *= i;\ni++;,i < n + 1,return result;,0
...,...,...,...,...,...,...,...,...,...,...,...
49491,49492,29,Write a JavaScript function that given a strin...,function capitalizeLetters(str) {\n let resul...,capitalizeLetters,str,"let result = "" "";\nlet i = 0;",result += str[i - 1];,i + 1 <= str.length,,7
49492,49493,29,Write a JavaScript function that given a strin...,function capitalizeLetters(str) {\n let i = s...,capitalizeLetters,str,let i = str.length - 1;\nlet result = [str];,result.unshift(str[i + 1]);,i + 1 < 0,,7
49493,49494,29,Write a JavaScript function that given a strin...,function capitalizeLetters(str) {\n let resul...,capitalizeLetters,str,"let result = ["" ""];\nlet i = str.length;",result.unshift(str[i]);,i - 1 > 0,return result;,7
49494,49495,29,Write a JavaScript function that given a strin...,function capitalizeLetters(str) {\n let i = 0...,capitalizeLetters,str,let i = 0;\nlet result = str;,result *= str[i].toUpperCase();,i < str.length,,7


In [5]:
result_df.to_csv("/content/drive/MyDrive/Universidad/Investigacion/dataframe_balanced_extracted.csv", index=False)